# T-CTRL, powered by ISCI — When a large cellular change is not control

**Researcher Track · Built with Claude: Life Sciences**

This executable walkthrough is for biologists, clinicians, and computational researchers. It explains the finding, reproduces its locked evidence from committed artifacts, and shows how a new Perturb-seq dataset enters the ISCI framework.

**Claim boundary:** ISCI does not declare therapeutic targets or clinical biomarkers. It tests whether a perturbation produces a directional, reproducible state shift beyond raw effect magnitude.

**Prerequisites:** basic familiarity with gene perturbation experiments and tabular data; no single-cell programming expertise is required. Install the repository with `uv sync --locked --extra dev` before executing.

**Learning goals:** distinguish perturbation size from state controllership, inspect all four evidence verdicts, and validate a new dataset contract without loading a large matrix.

**Outline:** (1) medical question, (2) supported and failed claims, (3) Claude correction loop, (4) reusable DatasetSpec workflow, (5) audit trail, and (6) prospective falsification.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository or notebooks directory."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from isci import load_dataset_spec, load_tabular_dataset, run_dataset, validate_dataset_spec

pd.set_option("display.max_colwidth", 90)


## 1. The medical question

A perturbation can make a cell change dramatically and still fail to control the clinically relevant state. The biological question is therefore not only **how much did the cell move?** but also where it moved, whether the direction repeated across donors and guides, and whether that information adds evidence after accounting for effect magnitude.

![Central controllership result](../figures/final/fig_central.png)


In [2]:
claims = json.loads((ROOT / "outputs/hackathon/claim_manifest.json").read_text())
primary = claims["claims"][0]
full = primary["metrics"]["authoritative_full_sample"]
oof = primary["metrics"]["leakage_free_oof"]

primary_metrics = pd.DataFrame([
    {
        "Estimand": "Full-sample pre-specified M → M+C",
        "ΔAUPRC": full["discovery_gain"],
        "95% CI": f'[{full["discovery_ci95"][0]:.3f}, {full["discovery_ci95"][1]:.3f}]',
        "Permutation p": None,
    },
    {
        "Estimand": "Leakage-free grouped OOF",
        "ΔAUPRC": oof["gain"],
        "95% CI": f'[{oof["ci95"][0]:.3f}, {oof["ci95"][1]:.3f}]',
        "Permutation p": oof["permutation_p"],
    },
])
display(primary_metrics)


,Estimand,ΔAUPRC,95% CI,Permutation p
0,Full-sample pre-specified M → M+C,0.3570,"[0.117, 0.538]",NaN
1,Leakage-free grouped OOF,0.2151,"[0.074, 0.560]",0.01


### Clinical-language interpretation

Among perturbations with comparable overall effect size, canonical T-cell state regulators were more likely to produce a **focused functional shift** and a **repeatable direction across donors**. The conservative out-of-fold estimate remained positive.

The two estimates answer related but different questions. They must not be averaged or presented as interchangeable.


## 2. A trustworthy result includes where it fails

The project uses four explicit verdicts. A failed generalization test does not erase a bounded primary result; it defines its scope. Missing inputs are not converted into a negative biological result.


In [3]:
verdict_rows = []
for claim in claims["claims"]:
    verdict_rows.append({
        "Question": claim["short_label"],
        "Verdict": claim["verdict"],
        "Scope": claim["scope"],
        "Do not claim": claim["prohibited_overclaim"],
    })
display(pd.DataFrame(verdict_rows))


,Question,Verdict,Scope,Do not claim
0,Canonical T-cell state controllers,PASS,Detectable-effect perturbations and canonical axis-defining T-cell regulators.,Broad functional-regulator discovery or therapeutic target nomination.
1,Broad external functional regulators,FAIL,Twenty external non-marker regulators with matched negatives.,The failed broad class invalidates the bounded canonical-regulator result.
2,CAR-T clinical response,NULL,Eighty-seven labeled patients across nine public studies.,Proof of no biological effect or a validated clinical biomarker.
3,scGPT zero-shot corroboration,NOT-EVALUABLE,Current machine state and the pre-specified expression-profile input contract.,"A scientific null, failed foundation model, or negative biological result."


### Stress test and scope map

![Positive-set stress test](../figures/positive_set_stress_test.png)

![Cross-system scope](../figures/cci_scope_4systems.png)


In [4]:
oof_evidence = json.loads((ROOT / "outputs/isci_oof_incremental.json").read_text())
stress = json.loads((ROOT / "outputs/positive_set_stress_test.json").read_text())
external = stress["external_screen_test"]

robustness = pd.DataFrame([
    {
        "Test": "Leakage-free OOF",
        "Result": "PASS",
        "ΔAUPRC": oof_evidence["oof_gain"],
        "95% CI": oof_evidence["hierarchical_bootstrap"]["ci95"],
        "p": oof_evidence["within_block_permutation"]["perm_p"],
    },
    {
        "Test": "Independent functional-regulator set",
        "Result": external["verdict"],
        "ΔAUPRC": external["gain"],
        "95% CI": external["ci"],
        "p": external["p_gain"],
    },
])
display(robustness)


,Test,Result,ΔAUPRC,95% CI,p
0,Leakage-free OOF,PASS,0.215100,"[0.0737, 0.5604]",0.010
1,Independent functional-regulator set,FAIL,-0.280728,"[-0.47638489435959946, -0.07342890735600327]",0.005


## 3. How Claude Science changed the research

Claude was not used to manufacture a positive result. It helped maintain a correction loop: formulate, challenge, falsify, reformulate, and bind each claim to evidence.


In [5]:
correction_history = pd.DataFrame([
    {"Stage": "Initial formulation", "Question": "Does a multi-component index beat magnitude?", "Outcome": "Failed: known regulators had much larger effects."},
    {"Stage": "Scientific correction", "Question": "Does direction/reproducibility add signal conditional on magnitude?", "Outcome": "Supported for canonical axis-defining regulators."},
    {"Stage": "Leakage audit", "Question": "Were matching and residualization repeated inside every fold?", "Outcome": "Pipeline corrected; OOF estimate decreased to a more honest +0.215."},
    {"Stage": "Generalization challenge", "Question": "Does the signal recover broad external functional regulators?", "Outcome": "No. The external test failed and bounded the claim."},
])
display(correction_history)


,Stage,Question,Outcome
0,Initial formulation,Does a multi-component index beat magnitude?,Failed: known regulators had much larger effects.
1,Scientific correction,Does direction/reproducibility add signal conditional on magnitude?,Supported for canonical axis-defining regulators.
2,Leakage audit,Were matching and residualization repeated inside every fold?,Pipeline corrected; OOF estimate decreased to a more honest +0.215.
3,Generalization challenge,Does the signal recover broad external functional regulators?,No. The external test failed and bounded the claim.


## 4. The reusable framework

The notebook is the teaching surface; the framework lives in typed Python interfaces and a versioned DatasetSpec. For a new dataset, `isci pipeline dataset.yaml` is the single entry point; every internal stage remains visible in its audit report.

**Dataset YAML → contract validation → physical preflight → matched-control effects → frozen feature runner → auditable outputs**

Supported execution boundary:

- **controller_features:** inspect and rank precomputed features;
- **long_effects:** extract features and rank from CSV or Parquet;
- **anndata_effects:** extract features and rank from backed H5AD with bounded-memory blocks;
- **anndata_cells:** preflight metadata, construct matched-control pseudobulk effects, and emit an `anndata_effects` spec.


In [6]:
example_path = ROOT / "examples/dataset_spec/mini_long_effects.yaml"
raw_spec = yaml.safe_load(example_path.read_text())
contract = validate_dataset_spec(raw_spec, repo_root=ROOT, check_paths=True)
spec = load_dataset_spec(example_path, repo_root=ROOT, check_paths=True)
physical = load_tabular_dataset(spec, repo_root=ROOT)
analysis = run_dataset(spec, repo_root=ROOT)

framework_check = pd.DataFrame([
    {
        "Layer": "DatasetSpec declaration",
        "Valid": contract.valid,
        "Capability": contract.capability.value,
        "Meaning": "Required fields are declared; data have not yet proven the claim.",
    },
    {
        "Layer": "Physical data inspection",
        "Valid": physical.inspection.evaluable,
        "Capability": physical.inspection.runtime_capability.value,
        "Meaning": "Observed coverage is too small for a confirmatory analysis.",
    },
    {
        "Layer": "Feature extraction + ranking",
        "Valid": analysis.completed,
        "Capability": analysis.status,
        "Meaning": "The tiny fixture lacks enough axis and replicate coverage; no ranking is invented.",
    },
])
display(framework_check)

cell_smoke = json.loads((ROOT / "outputs/hackathon/cell_preflight_smoke.json").read_text())["preflight"]
build_smoke = json.loads((ROOT / "outputs/hackathon/cell_effect_build_smoke.json").read_text())
cell_pipeline_summary = pd.DataFrame([
    {
        "Stage": "Metadata preflight",
        "Input": f'{cell_smoke["eligible_cells"]:,} cells × {cell_smoke["matrix_shape"][1]} proteins',
        "Output": f'{cell_smoke["eligible_effect_strata"]} supported strata',
        "Status": cell_smoke["status"],
        "Biological verdict": cell_smoke["biological_verdict"],
    },
    {
        "Stage": "Matched-control effect construction",
        "Input": f'{cell_smoke["perturbation_conditions_ready"]} perturbation units',
        "Output": f'{build_smoke["build"]["effect_rows"]} effect rows × {build_smoke["build"]["effect_features"]} proteins',
        "Status": build_smoke["status"],
        "Biological verdict": build_smoke["biological_verdict"],
    },
    {
        "Stage": "Frozen ISCI feature gate",
        "Input": "Generated anndata_effects spec",
        "Output": (
            f'{build_smoke["downstream"]["ranking_eligible_rows"]} ranking-eligible; '
            f'{build_smoke["downstream"]["missing_specificity_rows"]}/72 lack axis coverage; '
            f'{build_smoke["downstream"]["missing_reproducibility_rows"]}/72 lack reproducibility'
        ),
        "Status": build_smoke["downstream"]["runner_status"],
        "Biological verdict": build_smoke["biological_verdict"],
    },
])
display(cell_pipeline_summary)

arrayed_smoke = json.loads(
    (ROOT / "outputs/hackathon/arrayed_rna_effect_build_smoke.json").read_text()
)
external_path_comparison = pd.DataFrame([
    {
        "Public system": "THP-1 pooled protein",
        "Cells × features": f'{build_smoke["build"]["source_cells"]:,} × {build_smoke["build"]["effect_features"]:,}',
        "Effect rows": build_smoke["build"]["effect_rows"],
        "Ranking eligible": build_smoke["downstream"]["ranking_eligible_rows"],
        "Runner": build_smoke["downstream"]["runner_status"],
        "Why": "Four proteins do not cover the frozen CD4+ axes.",
    },
    {
        "Public system": "Jurkat arrayed RNA",
        "Cells × features": f'{arrayed_smoke["preflight"]["source_cells"]:,} × {arrayed_smoke["preflight"]["source_features"]:,}',
        "Effect rows": arrayed_smoke["build"]["effect_rows"],
        "Ranking eligible": arrayed_smoke["downstream"]["ranking_eligible_rows"],
        "Runner": arrayed_smoke["downstream"]["runner_status"],
        "Why": "Technical path complete; donor-free cell line remains diagnostic.",
    },
])
display(external_path_comparison)


,Layer,Valid,Capability,Meaning
0,DatasetSpec declaration,True,CONFIRMATORY_DECLARED,Required fields are declared; data have not yet proven the claim.
1,Physical data inspection,True,DIAGNOSTIC_ONLY,Observed coverage is too small for a confirmatory analysis.
2,Feature extraction + ranking,False,FEATURE_EXTRACTION_NOT_EVALUABLE,The tiny fixture lacks enough axis and replicate coverage; no ranking is invented.


,Stage,Input,Output,Status,Biological verdict
0,Metadata preflight,"20,729 cells × 4 proteins",216 supported strata,DIAGNOSTIC_ONLY,NOT_ISSUED
1,Matched-control effect construction,72 perturbation units,207 effect rows × 4 proteins,DIAGNOSTIC_EFFECTS_BUILT,NOT_ISSUED
2,Frozen ISCI feature gate,Generated anndata_effects spec,0 ranking-eligible; 72/72 lack axis coverage; 0/72 lack reproducibility,FEATURE_EXTRACTION_NOT_EVALUABLE,NOT_ISSUED


,Public system,Cells × features,Effect rows,Ranking eligible,Runner,Why
0,THP-1 pooled protein,"20,729 × 4",207,0,FEATURE_EXTRACTION_NOT_EVALUABLE,Four proteins do not cover the frozen CD4+ axes.
1,Jurkat arrayed RNA,"39,194 × 25,904",305,40,ANALYSIS_COMPLETE,Technical path complete; donor-free cell line remains diagnostic.


In [7]:
commands = pd.DataFrame([
    {"Goal": "Run the complete safe path", "Command": "isci pipeline dataset.yaml --output-dir outputs/my_dataset/pipeline"},
    {"Goal": "Validate contract and paths", "Command": "isci validate dataset.yaml"},
    {"Goal": "Validate before data download", "Command": "isci validate dataset.yaml --structure-only"},
    {"Goal": "Inspect CSV, Parquet, or H5AD", "Command": "isci inspect dataset.yaml"},
    {"Goal": "Scan H5AD layers by blocks", "Command": "isci inspect dataset.yaml --scan-values --block-rows 64"},
    {"Goal": "Preflight a cell-level H5AD", "Command": "isci preflight-cells dataset.yaml --report preflight.json"},
    {"Goal": "Build matched-control effects", "Command": "isci build-effects dataset.yaml --output-dir outputs/my_dataset/effects"},
    {"Goal": "Extract and rank any supported effect layout", "Command": "isci run dataset.effects.yaml --output-dir outputs/my_dataset/run"},
    {"Goal": "Limit H5AD matrix reads", "Command": "isci run dataset.yaml --block-rows 32"},
])
display(commands)


,Goal,Command
0,Run the complete safe path,isci pipeline dataset.yaml --output-dir outputs/my_dataset/pipeline
1,Validate contract and paths,isci validate dataset.yaml
2,Validate before data download,isci validate dataset.yaml --structure-only
3,"Inspect CSV, Parquet, or H5AD",isci inspect dataset.yaml
4,Scan H5AD layers by blocks,isci inspect dataset.yaml --scan-values --block-rows 64
5,Preflight a cell-level H5AD,isci preflight-cells dataset.yaml --report preflight.json
6,Build matched-control effects,isci build-effects dataset.yaml --output-dir outputs/my_dataset/effects
7,Extract and rank any supported effect layout,isci run dataset.effects.yaml --output-dir outputs/my_dataset/run
8,Limit H5AD matrix reads,isci run dataset.yaml --block-rows 32


### Exercise: bring a dataset

Copy the example DatasetSpec, change only paths and mappings, then predict the runtime capability before inspection. Record: **(a)** expected capability, **(b)** replicate unit, **(c)** axis coverage, and **(d)** the condition that should force NOT-EVALUABLE. A useful answer is not always confirmatory. DIAGNOSTIC_ONLY or NOT_EVALUABLE can be the scientifically correct outcome.


In [8]:
answer_scaffold = pd.Series({
    "Expected capability": "CONFIRMATORY / DIAGNOSTIC_ONLY / NOT-EVALUABLE because ...",
    "Biological replicate": "donor / guide / well / unavailable",
    "Axis coverage": "covered axes and missing required genes",
    "Stop condition": "the exact gate that should prevent a biological verdict",
}, name="Your prediction before inspection")
display(answer_scaffold.to_frame())

BYOD_SPEC = None  # Example: ROOT / "config/my_dataset.yaml"

if BYOD_SPEC is None:
    print("Set BYOD_SPEC to validate your own public Perturb-seq dataset. No external data were opened.")
else:
    candidate = load_dataset_spec(BYOD_SPEC, repo_root=ROOT, check_paths=True)
    print(f"Validated: {candidate.dataset.label} ({candidate.input.layout})")
    if candidate.input.layout == "anndata_cells":
        print("Next: run preflight-cells, inspect every exclusion, then build-effects locally.")
    else:
        print("Next: inspect every downgrade, then run isci run with a local output directory.")


,Your prediction before inspection
Expected capability,CONFIRMATORY / DIAGNOSTIC_ONLY / NOT-EVALUABLE because ...
Biological replicate,donor / guide / well / unavailable
Axis coverage,covered axes and missing required genes
Stop condition,the exact gate that should prevent a biological verdict


Set BYOD_SPEC to validate your own public Perturb-seq dataset. No external data were opened.


## 5. Auditability is part of the result

Every public claim is bound to a source artifact, data hash, axes hash, Git revision, command, and release gate. The framework never commits raw H5AD or H5MU inputs.


In [9]:
readiness = json.loads((ROOT / "outputs/hackathon/readiness_report.json").read_text())
readiness_summary = pd.DataFrame([
    {
        "Package status": readiness["status"],
        "Automated gates passed": f'{sum(readiness["checks"].values())}/{len(readiness["checks"])}',
        "Raw or secret tracked files": len(readiness["details"]["forbidden_tracked_files"]),
        "Public local-path violations": len(readiness["details"]["local_path_violations"]),
    }
])
display(readiness_summary)


,Package status,Automated gates passed,Raw or secret tracked files,Public local-path violations
0,AUTOMATED_GATES_PASS_HUMAN_GATES_PENDING,21/21,0,0


## 6. What is proven — and what is not

**Supported:** a magnitude-conditional signal for canonical axis-defining regulators in the primary CD4+ T-cell screen, with a positive leakage-free OOF estimate.

**Not supported:** broad functional-regulator generalization, clinical-response prediction, therapeutic desirability, or a universal immune-cell invariant.

**Framework boundary today:** one `isci pipeline` command accepts controller-feature tables, long-effect CSV/Parquet, backed effect-matrix H5AD, pooled cell-level H5AD, and arrayed single-guide H5AD. Every layout enters an audited path. A four-protein THP-1 input stopped at the axis gate; an independent 25,904-gene Jurkat RNA input completed all feature rows. Neither result was promoted beyond its evidence: successful preprocessing is not a biological validation, and donor-free cell-line evidence remains DIAGNOSTIC_ONLY.


## 7. Prospective experiment

The framework converts uncertainty into an experiment: a proposed 25-gene, 54-guide panel across 8–12 donors, with guide sequence and QC as a stop gate before synthesis.

![Controller convergence and prospective space](../figures/controller_convergence_quadrant.png)

The next scientific test is donor-resolved primary T-cell data: the same donors, matched controls, broad RNA coverage, and at least two guides per target. Selection must follow the frozen eligibility gates rather than a favorable outcome. Its output enters the same `anndata_effects` path without changing ISCI-core or the axes.


## Pitfall and optional extension

**Common pitfall:** treating thousands of cells or several guides as independent donors. The declared biological replicate controls the inference; more technical observations do not increase donor n.

**Optional extension:** create a new public DatasetSpec, run `isci pipeline`, and compare the diagnostic output with your pre-inspection prediction above. Do not promote the result unless every declared gate passes.

## Take-home message

**Magnitude tells us how far a cell moved. Controllership asks whether it moved in the intended functional direction, whether that direction repeated, and whether the evidence survives attempts to falsify it.**

T-CTRL is the auditable experience; ISCI is the bounded biological method for testing that question in new Perturb-seq datasets.
